In [ ]:
%load_ext autoreload

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "5"  # Limit OpenMP
os.environ["MKL_NUM_THREADS"] = "5"  # Limit MKL (Intel Math Kernel Library)
os.environ["OPENBLAS_NUM_THREADS"] = "5"  # Limit OpenBLAS
os.environ["NUMEXPR_MAX_THREADS"] = "5"  # Limit NumExpr if installed

In [ ]:
from collections import defaultdict
import itertools
from pathlib import Path
import re
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel
import seaborn as sns
import torch
from tqdm.auto import tqdm

In [ ]:
%autoreload 2
from src.data import get_electrode_df, add_metadata_features
from src.data_cleaning import prepare_ABC_results, compute_stimulus_correlation
from src.models.decoding import run_decoding_population, run_decoding_model_comparison_population

In [ ]:
epochs_paths = list(Path("outputs/epochs_preprocessed").glob("*.fif"))

result_paths = {
    k: list(Path("outputs/causal4/behavior_decoding_single_electrode_summarize").rglob(f"*/{k}_results.csv"))
    for k in ["A_early", "A"]
}

electrodes_paths = list(Path("outputs/causal4/find_speech_responsive").glob("*_results.csv"))

outdir = "."

# If an electrode shows a significant effect (p < this threshold) in any window, it will be included
# in the super-population
pval_inclusion_threshold = 0.05

decoding_smin = 0
decoding_tmax = 1.5
decoding_stride = 2
decoding_window_size = 15

In [ ]:
epochs = {
    str(re.search(r"(\w+)_epo.fif", str(path)).group(1)):
    mne.read_epochs(path, preload=True, verbose=False)
    for path in tqdm(epochs_paths)
}
for e in epochs.values():
    e.metadata = add_metadata_features(e.metadata)

In [ ]:
ep0 = next(iter(epochs.values()))
decoding_smax, = ep0.time_as_index([decoding_tmax])

In [ ]:
electrode_df = pd.concat([pd.read_csv(path) for path in electrodes_paths]).set_index(["subject", "electrode_idx"])

In [ ]:
res_dfs = {
    k: pd.concat([pd.read_csv(path) for path in result_paths[k]])
    for k in result_paths.keys()
}

In [ ]:
A_df = pd.concat([res_dfs["A_early"], res_dfs["A"]])

## Prepare super-populations

In [ ]:
ttest_df = A_df.groupby(["subject", "population", "phoneme_pair", "word_end", "smin", "smax"]) \
    .apply(lambda xs: pd.Series(ttest_rel(xs.baseline_roc_auc, xs.full_roc_auc), index=["tstat", "pval"]))
ttest_df["sig"] = (ttest_df.tstat < 0) & (ttest_df.pval < pval_inclusion_threshold)

In [ ]:
ttest_keep = ttest_df.groupby(["subject", "population", "phoneme_pair", "word_end"]).sig.any()
ttest_keep = ttest_keep[ttest_keep].reset_index().drop(columns="sig")
ttest_keep

## Estimate decoders

In [ ]:
super_decoding_results = {}
super_decoders = {}
super_populations = {}

In [ ]:
for (subject, word_end), rows in ttest_keep.groupby(["subject", "word_end"]):
    elec_idxs = rows.population.unique().tolist()
    if len(elec_idxs) <= 1:
        continue  # Need at least 2 electrodes to form a super-population
    assert rows.phoneme_pair.nunique() == 1
    phoneme_pair = rows.phoneme_pair.iloc[0]

    key = (subject, word_end)
    super_populations[key] = elec_idxs
    super_decoding_results[key], super_decoders[key] = run_decoding_model_comparison_population(
        epochs[subject],
        elec_idxs,
        phoneme_pair=phoneme_pair,
        subject=subject,
        population_name=f"super_{word_end}",
        stride=decoding_stride,
        window_size=decoding_window_size,
        global_min_sample=decoding_smin,
        global_max_sample=decoding_smax,
        pca_num_components=[0.1, 0.25, 0.35, 0.5, 0.9],
        target="behavior_categorical",
        baseline_features=["resampled"],
        strategy="train-test",
        groupby=["word_end"],
        filter=f"word_end == '{word_end}'",
        return_estimators=True,
        n_jobs=5,
    )

## Save

In [ ]:
torch.save({"super_decoding_results": super_decoding_results,
            "super_decoders": super_decoders,
            "super_populations": super_populations},
            f"{outdir}/results.pt")